In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import multiple_planets_gas_acc as code
from functions import *
%matplotlib inline
%load_ext autoreload 
%autoreload 2


In [ ]:
# disc parameters
params_dict = {'St_const': None, 
               'iceline_radius': None,
                'M_dot_gas_star': "Hartmann_2016",
                'star_mass': 1.0,
                'M_dot_star_scatter': True,
                }

params = code.Params(**params_dict)
fig, ax = plt.subplots(figsize=(8, 6))
t = np.logspace(-1, 2, 100)  # Time from 0.1 Myr to 100 Myr
mdot = (M_dot_star(t, params)*u.M_earth/u.Myr).to(u.Msun/u.yr)
mdot_photoevap = (M_dot_photoevaporation(params)*u.M_earth/u.Myr).to(u.Msun/u.yr).value
mdot_photo = np.full_like(t, mdot_photoevap)  # Constant photoevaporation rate
ax.loglog(t, mdot, label='Gas Accretion Rate (M_dot_star)', color='blue')
ax.loglog(t, mdot_photo, label='Photoevaporation Rate (M_dot_photo)', color='red')
ax.set_xlabel('Time (Myr)')
ax.set_ylabel('Rate ($M_{\odot}$/yr)')
ax.set_title('Gas Accretion and Photoevaporation Rates')
ax.legend()


In [ ]:
# disc parameters
params_dict = {'St_const': None, 
               'iceline_radius': None,
                'star_mass': 1*(const.M_sun).to(u.M_earth).value,
                'M_dot_star_scatter': True,
                }


mdots = ["Hartmann_2016", "Manara_2012", "star_mass_linear", "star_mass_quadratic"]
colors = ['blue', 'orange', 'green', 'red']
fig, ax = plt.subplots(figsize=(8, 6))
t = np.geomspace(0.1, 10, 100)  # Time from 0.1 Myr to 100 Myr
for i, mdot_name in enumerate(mdots):
    mdot_tot = []
    params = code.Params(**params_dict, M_dot_gas_star = mdot_name)
    mdot = (M_dot_star(t, params)*u.M_earth/u.Myr).to(u.Msun/u.yr).value
    mdot_photoevap = (M_dot_photo(params)*u.M_earth/u.Myr).to(u.Msun/u.yr).value
    mdot_photo = np.full_like(t, mdot_photoevap)  # Constant photoevaporation rate
    for time in t:
        mdot_tot.append((M_dot_star_t_photoevap(time, r_magnetic_cavity(M_dot_star(0.1, params), params), params)*u.M_earth/u.Myr).to(u.Msun/u.yr).value)
    print("mdot_tot for {}: {}".format(mdot_name, mdot_tot))
    print("mdot shape", len(mdot_tot))
    ax.loglog(t, mdot_tot, label=params.M_dot_gas_star + ' total' if i == 0 else None, linestyle=':', color=colors[i])
    ax.loglog(t, mdot, label=params.M_dot_gas_star, color=colors[i])
    ax.loglog(t, mdot_photo, linestyle='--', color=colors[i])
    ax.axvline(x=t_photo(params), color=colors[i])

ax.set_xlabel('Time (Myr)')
ax.set_ylabel('Rate ($M_{\odot}$/yr)')
ax.set_title('Gas Accretion and Photoevaporation Rates')
ax.legend()
mdostar = M_dot_star(0.1, params) #here I do not use the one that has photevap cause at the beginning is not dominant
R_in = r_magnetic_cavity(mdostar, params)
for mdot_name in mdots:
    params = code.Params(**params_dict, M_dot_gas_star = mdot_name)
    t_photoevap = t_photo(params)
    tau_photoevap = tau_photo(R_in, params)
    print("photoevaporation onset for {}: {:.2f} Myr".format(mdot_name, t_photoevap))
    print("photoevaporation timescale for {}: {:.2f} Myr".format(mdot_name, tau_photoevap))
plt.savefig('figures/gas_accretion_photoevaporation_rates_newfunc_scatter.png', dpi=300)

In [ ]:
for model in ["Hartmann_2016", "Manara_2012", "star_mass_linear", "star_mass_quadratic"]:
    params.M_dot_gas_star = model
    t0 = t_photo(params)
    print(model, "t_ph", t0, "ratio", M_dot_star(t0, params)/M_dot_photo(params))

In [ ]:
mdot_star = (1e-7*u.M_sun/u.yr).to(u.M_earth/u.Myr).value
R_disc = 71  # in AU
params = code.Params(**params_dict, M_dot_gas_star = "Hartmann_2016")
M_disc = M_disc_t_irr(R_disc, mdot_star, params)
M_disc_visc = M_disc_t_visc(R_disc, mdot_star, params)
M_E_to_M_sun = (u.M_earth).to(u.M_sun)
print(f"Disc mass at R_disc = {R_disc} AU and M_dot_star = {mdot_star:.2e} M_earth/Myr, and M_dot_star = {mdot_star*M_E_Myr_to_M_sun_yr:.2e} M_sun/yr:, irradiated disc {M_disc*M_E_to_M_sun:.2e} M_sun")
print(f"Disc mass at R_disc = {R_disc} AU and M_dot_star = {mdot_star:.2e} M_earth/Myr, and M_dot_star = {mdot_star*M_E_Myr_to_M_sun_yr:.2e} M_sun/yr:, viscous disc {M_disc_visc*M_E_to_M_sun:.2e} M_sun")

In [ ]:
t = 0.1
mdot_star = M_dot_star(t, params)
print("alpha", params.alpha)
print("transittion radius for M_dot_star =", mdot_star*M_E_Myr_to_M_sun_yr,"M_sun/yr, with disc",params.H_r_model, 
"\nwith epsilon el",params.epsilon_el,"and epsilon heat",params.epsilon_heat,":",H_r_transition_radius(mdot_star, params))